In [0]:
datasets = {
    'daily_production': spark.read.parquet('/Volumes/workspace/default/raw_data/daily_production.parquet'),
    'production_failure': spark.read.parquet('/Volumes/workspace/default/raw_data/production_failures.parquet'),
    'well_jobs': spark.read.parquet('/Volumes/workspace/default/raw_data/well_jobs.parquet'),
}

for name, df in datasets.items():
  df.createOrReplaceTempView(name)

In [0]:
%sql
--DISPLAY FIRST 20 ROWS OF DATASET (daily_production)
SELECT * FROM daily_production LIMIT 10;

In [0]:
%sql
--DISPLAY FIRST 10 ROWS OF DATASET (production_failures)
SELECT * FROM production_failure LIMIT 20;
    

In [0]:
%sql
--DISPLAY FIRST 10 ROWS OF DATASET (well_jobs)
SELECT * FROM well_jobs LIMIT 10; 

In [0]:
%sql
-- pivot table showing the number of each job_category and total job done for each uwi
WITH cleaned AS (
    SELECT
        UPPER(TRIM(uwi)) AS uwi_clean,
        TRIM(job_category) AS job_category_clean
    FROM well_jobs
    WHERE uwi IS NOT NULL
      AND TRIM(uwi) <> ''
      AND UPPER(TRIM(uwi)) NOT IN ('NA', 'N/A', 'NULL', 'NONE')
)
SELECT
    uwi_clean AS uwi,
    COUNT(*) FILTER (WHERE LOWER(job_category_clean) = 'rod truck') AS rod_truck_count,
    COUNT(*) FILTER (WHERE LOWER(job_category_clean) = 'workover') AS workover_count,
    COUNT(*) FILTER (WHERE LOWER(job_category_clean) = 'drilling') AS drilling_count,
    COUNT(*) FILTER (WHERE LOWER(job_category_clean) = 'completion') AS completion_count,
    COUNT(*) FILTER (WHERE LOWER(job_category_clean) = 'abandon') AS abandon_count,
    COUNT(*) FILTER (
        WHERE job_category_clean IS NULL
           OR job_category_clean = ''
           OR LOWER(job_category_clean) NOT IN ('rod truck', 'workover', 'drilling', 'completion', 'abandon')
    ) AS other_or_unspecified_count,
    COUNT(*) AS total_jobs
FROM cleaned
GROUP BY uwi_clean
ORDER BY total_jobs DESC, uwi_clean;

In [0]:
%sql
--
WITH classified_jobs AS (
    -- same CTE as above
    SELECT
        UPPER(TRIM(uwi)) AS uwi_clean,
        CASE
            WHEN LOWER(job_type) LIKE '%polish rod%' OR LOWER(job_type) LIKE '%parted rod%' 
            OR LOWER(job_type) LIKE '%broken rod%' 
            OR LOWER(job_type) LIKE '%rod parted%' 
            OR LOWER(job_type) LIKE '%rod break%' 
            OR LOWER(job_type) LIKE '%sucker rod%'
                THEN 'Rods / Polish Rod'
            WHEN LOWER(job_type) LIKE '%tubing leak%' 
            OR LOWER(job_type) LIKE '%tubing split%' 
            OR LOWER(job_type) LIKE '%hole in tbg%' OR LOWER(job_type) LIKE '%tbg leak%' 
            OR LOWER(job_type) LIKE '%tbg split%' 
            OR LOWER(job_type) LIKE '%corroded tbg%'
                THEN 'Tubing Leak / Split'
            WHEN LOWER(job_type) LIKE '%stator%' 
            OR LOWER(job_type) LIKE '%rotor%' 
            OR LOWER(job_type) LIKE '%pump wear%' 
            OR LOWER(job_type) LIKE '%worn pump%' 
            OR LOWER(job_type) LIKE '%seized%' 
            OR LOWER(job_type) LIKE '%stuck pump%' 
            OR LOWER(job_type) LIKE '%pump change%'
                THEN 'Pump / Rotor / Stator'
            WHEN LOWER(job_type) LIKE '%packer%' 
            OR LOWER(job_type) LIKE '%bridge plug%' 
            OR LOWER(job_type) LIKE '%cbp%' 
            OR LOWER(job_type) LIKE '%pbp%' 
            OR LOWER(job_type) LIKE '%plug%'
                THEN 'Packer / Bridge Plug'
            WHEN LOWER(job_type) LIKE '%sand%' 
            OR LOWER(job_type) LIKE '%sanded%' 
            OR LOWER(job_type) LIKE '%sand fill%' 
            OR LOWER(job_type) LIKE '%cleanout%' 
            OR LOWER(job_type) LIKE '%sco%'
                THEN 'Sand / Fill / Cleanout'
            ELSE 'Other / Unclassified'
        END AS job_type
    FROM well_jobs
    WHERE uwi IS NOT NULL AND TRIM(uwi) <> ''
)
SELECT
    uwi_clean AS uwi,
    COUNT(*) FILTER (WHERE job_type = 'Rods / Polish Rod') AS rods_polish_rod,
    COUNT(*) FILTER (WHERE job_type = 'Tubing Leak / Split') AS tubing_leak_split,
    COUNT(*) FILTER (WHERE job_type = 'Pump / Rotor / Stator') AS pump_rotor_stator,
    COUNT(*) FILTER (WHERE job_type = 'Packer / Bridge Plug') AS packer_bridge_plug,
    COUNT(*) FILTER (WHERE job_type = 'Sand / Fill / Cleanout') AS sand_fill_cleanout,
    COUNT(*) FILTER (WHERE job_type = 'Other / Unclassified') AS other_unclassified,
    COUNT(*) AS total_jobs
FROM classified_jobs
GROUP BY uwi_clean
ORDER BY total_jobs DESC, uwi_clean;

In [0]:
%sql
-- average cost of maintainence and maintanence duration per pump make and model 
WITH distinct_wells AS (
    SELECT DISTINCT
        uwi,
        pcp_pump_make AS make,
        pcp_pump_model AS model
    FROM daily_production
    WHERE uwi IS NOT NULL
),
well_job_est AS (
    SELECT
        dw.uwi,
        dw.make,
        dw.model,
        AVG(wj.field_est_total) AS avg_est_per_uwi,
        AVG(wj.job_duration_days) AS avg_job_duration
    FROM well_jobs AS wj
    INNER JOIN distinct_wells AS dw
        ON wj.uwi = dw.uwi
    WHERE wj.field_est_total IS NOT NULL
    GROUP BY
        dw.uwi,
        dw.make,
        dw.model
)
SELECT
    make,
    model,
    ROUND(AVG(avg_est_per_uwi),1) AS avg_estimate_cost_total,
    ROUND(AVG(avg_job_duration)) AS avg_job_duration
FROM well_job_est
GROUP BY
    make,
    model
ORDER BY
    avg_estimate_cost_total ASC,
    avg_job_duration,
    make,
    model
    LIMIT 10;

In [0]:
%sql
-- 
WITH well_production AS (
    SELECT
        uwi,
        ROUND(SUM(oil_produced_m3), 3) AS total_oil_produced,
        ROUND(SUM(gas_produced_e3m3), 3) AS total_gas_produced,
        ROUND(SUM(lost_oil_m3), 3) AS total_lost_oil,
        ROUND(SUM(oil_produced_m3 - lost_oil_m3), 3) AS net_gain_m3
    FROM daily_production
    GROUP BY uwi
),

pump_info AS (
    SELECT DISTINCT
        uwi,
        pcp_pump_make,
        pcp_pump_model,
        pump_efficiency_pct
    FROM daily_production
)

SELECT
    p.uwi,
    w.pcp_pump_make,
    w.pcp_pump_model,
    p.total_oil_produced,
    p.total_gas_produced,
    p.total_lost_oil,
    p.net_gain_m3,
    ROUND(w.pump_efficiency_pct, 3) AS pump_efficientcy_pct
FROM well_production p
LEFT JOIN pump_info w
    ON p.uwi = w.uwi
ORDER BY p.net_gain_m3 DESC
LIMIT 15;

In [0]:
%sql
-- Top 5 pump models average efficiency
SELECT
    pcp_pump_make,
    pcp_pump_model,
    ROUND(AVG(pump_efficiency_pct), 3) AS avg_efficiency_pct
FROM daily_production
WHERE pump_efficiency_pct IS NOT NULL
GROUP BY
    pcp_pump_make,
    pcp_pump_model
ORDER BY avg_efficiency_pct DESC
LIMIT 10;

In [0]:
%sql
-- Combined maintenance cost/duration and pump efficiency per pump make and model
WITH distinct_wells AS (
    SELECT DISTINCT
        uwi,
        pcp_pump_make AS make,
        pcp_pump_model AS model
    FROM daily_production
    WHERE uwi IS NOT NULL
      AND pcp_pump_make IS NOT NULL
      AND pcp_pump_model IS NOT NULL
),
well_job_est AS (
    SELECT
        dw.uwi,
        dw.make,
        dw.model,
        AVG(wj.field_est_total) AS avg_est_per_uwi,
        AVG(wj.job_duration_days) AS avg_job_duration
    FROM well_jobs AS wj
    INNER JOIN distinct_wells AS dw
        ON wj.uwi = dw.uwi
    WHERE wj.field_est_total IS NOT NULL
    GROUP BY
        dw.uwi,
        dw.make,
        dw.model
),
maint_summary AS (
    SELECT
        make,
        model,
        ROUND(AVG(avg_est_per_uwi), 1) AS avg_estimate_cost_total,
        ROUND(AVG(avg_job_duration)) AS avg_job_duration
    FROM well_job_est
    GROUP BY
        make,
        model
),
pump_efficiency AS (
    SELECT
        pcp_pump_make AS make,
        pcp_pump_model AS model,
        ROUND(AVG(pump_efficiency_pct), 3) AS avg_efficiency_pct
    FROM daily_production
    WHERE pump_efficiency_pct IS NOT NULL
      AND pcp_pump_make IS NOT NULL
      AND pcp_pump_model IS NOT NULL
    GROUP BY
        pcp_pump_make,
        pcp_pump_model
)
SELECT
    COALESCE(ms.make, pe.make) AS make,
    COALESCE(ms.model, pe.model) AS model,
    ms.avg_estimate_cost_total,
    ms.avg_job_duration,
    pe.avg_efficiency_pct
FROM maint_summary AS ms
FULL OUTER JOIN pump_efficiency AS pe
    ON ms.make = pe.make
   AND ms.model = pe.model
ORDER BY
    pe.avg_efficiency_pct DESC NULLS LAST,
    ms.avg_estimate_cost_total ASC NULLS LAST,
    make ASC,
    model ASC;

In [0]:
%sql
SELECT
    province,
    pcp_pump_make,
    pcp_pump_model,
    COUNT(DISTINCT uwi) AS total_pumps
FROM daily_production
WHERE province IS NOT NULL
  AND pcp_pump_make IS NOT NULL
  AND pcp_pump_model IS NOT NULL
  AND uwi IS NOT NULL
GROUP BY
    province,
    pcp_pump_make,
    pcp_pump_model
ORDER BY
    province ASC,
    total_pumps DESC,
    pcp_pump_make ASC,
    pcp_pump_model ASC;

In [0]:
%sql
--Total pump in each province
SELECT
    province,
    COUNT(DISTINCT uwi) AS total_wells
FROM daily_production
WHERE province IS NOT NULL
  AND uwi IS NOT NULL
GROUP BY
    province
ORDER BY
    total_wells DESC,
    province ASC;

In [0]:
%sql
--Each province production
SELECT
    province,
    SUM(oil_produced_m3) AS total_oil_produced,
    SUM(gas_produced_e3m3) AS total_gas_produced,
    SUM(lost_oil_m3) AS total_lost_oil,
    SUM(oil_produced_m3 - lost_oil_m3) AS net_gain_m3
FROM daily_production
WHERE province IS NOT NULL
GROUP BY province
ORDER BY total_oil_produced DESC
LIMIT 10;
    

In [0]:
%sql
--